# nano-AgentRL on a free Kaggle T4

Train a 0.5B model with GRPO to **reason, search Wikipedia, and answer** — the
[Search-R1](https://arxiv.org/abs/2503.09516) setup, re-implemented from scratch in
[`PRINCE2-AI/nano-agentrl`](https://github.com/PRINCE2-AI/nano-agentrl).

**Notebook settings:** Accelerator *GPU T4 x2*, Internet *on*, Persistence *Files only*.

**Runtime:** ~5 min retriever setup + ~3 h training. Use *Save Version → Save & Run All* so
the run continues with the browser closed; re-running a cell resumes from the latest
checkpoint.

**What you should see:** for the first ~45 steps the model takes a shortcut (answers
immediately, never searches). Around step 56 it discovers retrieval, searches once per
question, and NQ / HotpotQA exact match moves from ~0 to 13 / 20%.

> Only 0.5B fits here. The 1.5B / 3B configs in `configs/` need an 80 GB GPU.

## 0. Check the GPU
If this fails or shows one T4, set *Accelerator → GPU T4 x2* in the notebook settings and restart.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

## 1. Code and dependencies
Clones the repo on the first run. `.[dev]` pulls torch, transformers, matplotlib and the test deps.

In [ ]:
import os

if not os.path.isdir('nano-agentrl'):
    !git clone https://github.com/PRINCE2-AI/nano-agentrl
%cd nano-agentrl
!pip install -q -e ".[dev]"
!git log --oneline -1

## 2. 30-second sanity check
178 CPU tests, no downloads. If this fails, nothing below will work — read the failure first.

In [ ]:
!pytest -q

## 3. BM25 retriever (~5 min)
Downloads the wiki-18 index and corpus, installs Java 21 + Pyserini, then starts the server
as a detached process on `127.0.0.1:8000`. Logs: `/tmp/retriever.log`.

In [ ]:
!bash scripts/kaggle/setup_retriever.sh
!python scripts/kaggle/start_retriever.py

## 4. Train the search agent (~3 h)
90 steps, 8 rollouts per question, 2 turns, 300 tokens per turn. Metrics land in
`runs/search_0.5b_t4/metrics.jsonl`; checkpoints in `runs/search_0.5b_t4/`.

Watch `train/num_searches`: flat at ~0 means the model is still in the format shortcut, not that
training is broken.

In [ ]:
!pip install -q bitsandbytes
!python scripts/train.py --config configs/search_0.5b_t4.yaml

## 5. Evaluate the checkpoint
Greedy EM on 500 NQ + 500 HotpotQA questions, plus the lenient gold-in-answer metric. Pass
`--ckpt base` to score the untrained model for comparison.

In [ ]:
import glob

ckpts = sorted(glob.glob('runs/search_0.5b_t4/checkpoint-*'))
print(ckpts[-1] if ckpts else 'no checkpoint yet')

In [ ]:
!python scripts/train.py --config configs/search_0.5b_t4.yaml --eval_only --ckpt $(ls -d runs/search_0.5b_t4/checkpoint-* | tail -1) --set eval_num_examples=500

## 6. Reliability sweep (optional, ~1 h)
8 samples per question at temperature 1.0, on the 100-question subsets. This is where the
repo's headline result comes from: **pass@8 21–28% but pass^8 3–4%** — the model answers a
quarter of the questions *sometimes* and almost none *every time*.

No `--use_vllm` here (vLLM is not installed); sampling with HF is slower but identical in output.

In [ ]:
!python scripts/reliability_sweep.py --run_dir runs/search_0.5b_t4 --split nq_dev_500 hotpotqa_dev_500 --max_examples 100 --n_samples 8 --max_turns 2 --max_new_tokens 300 --judge rule

## 7. Retrieval recall@3 (optional, ~2 min)
~70% of failures never retrieve the gold passage. This reports whether the raw question or
the agent's own query finds it, and the ceiling if the agent kept searching — so a low score
points at BM25 or the corpus, not the policy.

In [ ]:
# Retriever or policy? Ask BM25 for the gold passage twice: with the raw question, and
# with the query the agent actually wrote (sample 0, step 90 of the sweep above).
!python scripts/retrieval_recall.py --split nq_dev_500 --max_examples 100 --records runs/search_0.5b_t4/reliability/90/nq_dev_500.jsonl

## 8. Figures
Writes PNG + PDF into `docs/figures/search_0.5b_t4/`. Everything you see in the README comes
from these files.

In [ ]:
!python scripts/plot.py --metrics runs/search_0.5b_t4/metrics.jsonl --reliability runs/search_0.5b_t4/reliability.csv --out_dir docs/figures/search_0.5b_t4 --split nq_dev_500

## Troubleshooting

- **`java not found`** — re-run cell 3; the JDK install has to finish before the server starts.
- **Retriever already running** — free T4 sessions are recycled; a stale `/tmp/retriever.log`
  from a previous session is harmless, but a hung server is not. Restart the session.
- **CUDA OOM during training** — lower `micro_batch_size` in
  `configs/search_0.5b_t4.yaml`; GRPO's group size stays the same, only the step count changes.
- **Results look worse than the README** — check `train/num_searches`. If the model never
  searched, it is stuck in the format shortcut; that run needs more steps, not a different metric.